### Creates responses using the new iRAT pipeline

In [1]:
# Get answer for each query and store in a file.
import json
import os

from irat.pipeline import run_pipeline
from irat.utils.common_functions import print_separator
from irat.utils.logger import log_info
from irat.utils.ratelimit_counter import wait_for_rate_limit

# ds_name = 'human_eval'
ds_name = 'mbpp'
# ds_name = 'gsm8k'

if ds_name == 'mbpp':
	ds_rows = []
	# https://github.com/google-research/google-research/blob/master/mbpp/sanitized-mbpp.json
	dataset_file = os.path.join('irat', 'processed', f'mbpp_github.json')
	with open(dataset_file) as file:
		rows = json.load(file)
		for row in rows:
			# These indices are evaluated in old RAT paper.
			if 11 <= row['task_id'] <= 175:
				sample_test_cases = '\n'.join(row['test_imports'] + row['test_list'])
				prompt = (
					f'{row["prompt"]}\n\n'
					'Here are some sample test cases. Use the same function name and arguments.\n'
					f'{sample_test_cases}'
				)
				ds_rows.append({
					'task_id': row['task_id'],
					'prompt': prompt,
					'original_prompt': row['prompt'],
					'test_imports': row['test_imports'],
					'test_list': row['test_list'],
					'mbpp_code': row['code'],
				})
elif ds_name == 'human_eval':
	ds_rows = []
	from human_eval.data import read_problems
	problems = read_problems()
	for prob in problems:
		ds_rows.append({
			'task_id': prob,
			'prompt': problems[prob]['prompt'],
		})
elif ds_name == 'gsm8k':
	ds_rows = []
	# https://raw.githubusercontent.com/openai/grade-school-math/refs/heads/master/grade_school_math/data/test.jsonl
	dataset_file = os.path.join('irat', 'processed', f'{ds_name}_testdata_github.jsonl')
	with open(dataset_file) as file:
		for line in file:
			data = json.loads(line)
			ds_rows.append({
				'task_id': len(ds_rows) + 1,
				'prompt': data['question'],
				'correct_answer': data['answer'],
			})
			if len(ds_rows) > 100:
				break

# If present in project home, switch to the samples directory
if os.path.exists('README.md'):
	dir = f'{ds_name}-responses'
	os.chdir('evaluation')
	if not os.path.exists(dir):
		os.makedirs(dir)
	os.chdir(dir)

2025-06-25 12:46:47 [INFO] Testing the evaluator...
2025-06-25 12:46:47 [INFO] Getting feedback...


In [2]:
# Get a file name that is available to use. Attempt to use test_1.json, test_2, etc.
end = 20
for index in range(100_000):
	if index >= end or index > len(ds_rows):
		break
	out_file = f'test_{index+1}.json'
	if os.path.exists(out_file):
		log_info(f'File {out_file} already exists. Skipping.')
		print_separator()
		continue
	query = ds_rows[index]['prompt'].strip()
	log_info(f'Processing query {index+1}: {query}')
	response_result = run_pipeline(query)
	if response_result is None:
		log_info(f'No response for query {index+1} which may be due to unsafe query. Skipping.')
		print_separator()
		continue
	draft_1, draft_2, all_revisions, evaluator_feedback, final_answer, \
		all_retrievals, total_time = response_result
	with open(out_file, 'w') as file:
		json.dump({
			'task_id': ds_rows[index]['task_id'],
			'query': query,
			'draft_1': draft_1,
			'draft_2': draft_2,
			'evaluator_feedback': evaluator_feedback,
			'final_answer': final_answer,
			'all_retrievals': all_retrievals,
			'total_time': total_time,
			'all_revisions': all_revisions,  # only for reference
			# copy all other fields from ds_rows[index] if needed
			**ds_rows[index],
		}, file, indent=4)
	log_info(f'Saved results to {out_file}')
	print_separator()
	wait_for_rate_limit(10)  # To avoid rate limits.
log_info('All queries processed. Created JSON files.')

2025-06-25 12:46:53 [INFO] Processing query 1: Write a python function to remove first and last occurrence of a given character from the string.

Here are some sample test cases. Use the same function name and arguments.
assert remove_Occ("hello","l") == "heo"
assert remove_Occ("abcda","a") == "bcd"
assert remove_Occ("PHP","P") == "H"
2025-06-25 12:46:53 [INFO] Generating draft v1...
.
2025-06-25 12:47:00 [INFO] Fetched the Draft
2025-06-25 12:47:00 [INFO] Generating draft v2...
2025-06-25 12:47:00 [INFO] Processing Drafts...
2025-06-25 12:47:00 [INFO] The draft is divided into 6 parts
---------- - 0 -  - 0 -  - 0 -  - 0 -  - 0 -  - 0 -  - 0 -  - 0 -  - 0 -  - 0 - ----------
2025-06-25 12:47:00 [INFO] Modify 1/6 parts...
2025-06-25 12:47:00 [INFO] Generating corresponding Query...
.
>>> 0/6 Query: python function remove first and last occurrence of given character from string with examples
2025-06-25 12:47:01 [INFO] Get web page content...
2025-06-25 12:47:03 [INFO] Filtering retrieved

Fetching pages: 100%|##########| 1/1 [00:08<00:00,  8.66s/it]


2025-06-25 12:47:18 [INFO] Filtered paragraphs: 6
2025-06-25 12:47:18 [INFO] Modifying the answer according to page...[1/3]
.
2025-06-25 12:47:24 [INFO] Answer updation completed: [1/3]
2025-06-25 12:47:24 [INFO] Modifying the answer according to page...[2/3]
.
2025-06-25 12:47:30 [INFO] Answer updation completed: [2/3]
2025-06-25 12:47:30 [INFO] Modifying the answer according to page...[3/3]
.
2025-06-25 12:47:35 [INFO] Answer updation completed: [3/3]
---------- - 1 -  - 1 -  - 1 -  - 1 -  - 1 -  - 1 -  - 1 -  - 1 -  - 1 -  - 1 - ----------
2025-06-25 12:47:35 [INFO] Modify 2/6 parts...
2025-06-25 12:47:35 [INFO] Generating corresponding Query...
.
>>> 1/6 Query: python remove first and last occurrence of character in string code example "if first_index == -1: return s"
2025-06-25 12:47:37 [INFO] Get web page content...
2025-06-25 12:47:38 [INFO] Filtering retrieved results...
Using URLs: []
2025-06-25 12:47:38 [ERROR] Error in get_content: No URLs provided for filtering.
2025-06-25 

Fetching pages: 100%|##########| 1/1 [00:02<00:00,  2.11s/it]

Fetching pages: 100%|##########| 1/1 [00:02<00:00,  2.83s/it]


2025-06-25 12:47:49 [INFO] Filtered paragraphs: 5
2025-06-25 12:47:49 [INFO] Modifying the answer according to page...[1/3]
.
2025-06-25 12:47:56 [INFO] Answer updation completed: [1/3]
2025-06-25 12:47:56 [INFO] Modifying the answer according to page...[2/3]
.
2025-06-25 12:48:00 [INFO] Answer updation completed: [2/3]
2025-06-25 12:48:00 [INFO] Modifying the answer according to page...[3/3]
.
2025-06-25 12:48:07 [INFO] Answer updation completed: [3/3]
---------- - 3 -  - 3 -  - 3 -  - 3 -  - 3 -  - 3 -  - 3 -  - 3 -  - 3 -  - 3 - ----------
2025-06-25 12:48:07 [INFO] Modify 4/6 parts...
2025-06-25 12:48:07 [INFO] Generating corresponding Query...
.
>>> 3/6 Query: python remove first and last occurrence of character string slicing order
2025-06-25 12:48:09 [INFO] Get web page content...
2025-06-25 12:48:10 [INFO] Filtering retrieved results...
Using URLs: ['https://discuss.python.org/t/removing-all-but-one-instance-of-a-phrase-from-a-string/49954', 'https://community.esri.com/t5/pytho

...

Fetching pages: 100%|##########| 1/1 [00:06<00:00,  6.33s/it]


2025-06-25 12:48:55 [INFO] Filtered paragraphs: 7
2025-06-25 12:48:55 [INFO] Modifying the answer according to page...[1/3]
.
2025-06-25 12:49:00 [INFO] Answer updation completed: [1/3]
2025-06-25 12:49:00 [INFO] Modifying the answer according to page...[2/3]
.
2025-06-25 12:49:05 [INFO] Answer updation completed: [2/3]
2025-06-25 12:49:05 [INFO] Modifying the answer according to page...[3/3]
.
2025-06-25 12:49:09 [INFO] Answer updation completed: [3/3]
---------- - 5 -  - 5 -  - 5 -  - 5 -  - 5 -  - 5 -  - 5 -  - 5 -  - 5 -  - 5 - ----------
2025-06-25 12:49:09 [INFO] Modify 6/6 parts...
2025-06-25 12:49:09 [INFO] Generating corresponding Query...
.
>>> 5/6 Query: python function remove first and last occurrence of character from string with test cases
2025-06-25 12:49:10 [INFO] Get web page content...
2025-06-25 12:49:11 [INFO] Filtering retrieved results...
Using URLs: ['https://discuss.python.org/t/removing-all-but-one-instance-of-a-phrase-from-a-string/49954', 'https://superuser.c

Fetching pages: 100%|##########| 1/1 [00:06<00:00,  6.41s/it]


2025-06-25 12:49:23 [INFO] Filtered paragraphs: 3
2025-06-25 12:49:23 [INFO] Modifying the answer according to page...[1/3]
.
2025-06-25 12:49:28 [INFO] Answer updation completed: [1/3]
2025-06-25 12:49:28 [INFO] Modifying the answer according to page...[2/3]
.
2025-06-25 12:49:35 [INFO] Answer updation completed: [2/3]
2025-06-25 12:49:35 [INFO] Modifying the answer according to page...[3/3]
.
2025-06-25 12:49:42 [INFO] Answer updation completed: [3/3]
2025-06-25 12:49:42 [INFO] Reflection: Analyzing the thoughts...
2025-06-25 12:49:42 [INFO] Getting feedback...
2025-06-25 12:50:11 [INFO] Generating draft v3...
.
2025-06-25 12:50:16 [INFO] Saved results to test_1.json
--------------------------------------------------------------------------------
2025-06-25 12:50:26 [INFO] Processing query 2: Write a function to sort a given matrix in ascending order according to the sum of its rows.

Here are some sample test cases. Use the same function name and arguments.
assert sort_matrix([[1, 2

KeyboardInterrupt: 